---
## Section 1: Imports & Device Setup

In [ ]:
import sys
!{sys.executable} -m pip install -q torch torchvision medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 1.3 MB/s eta 0:00:00


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, ConcatDataset
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from medmnist import PneumoniaMNIST, BreastMNIST, DermaMNIST, PathMNIST, RetinaMNIST

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

np.random.seed(42); torch.manual_seed(42)
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')
if DEVICE.type == 'cpu':
    print('WARNING: running on CPU. This will be slow. Check Runtime > Change runtime type > GPU.')

Device  : cpu
PyTorch : 2.11.0+cpu


---
## Section 2: Configuration and Label Mapping

In [ ]:
CONFIG = {
    'num_rounds'          : 8,
    'local_epochs'        : 4,
    'batch_size'          : 64,
    'learning_rate'       : 0.0005,
    'image_size'          : 96,
    'mu_prox'             : 0.01,
    'server_lr'           : 0.1,
    'beta1'               : 0.9,
    'beta2'               : 0.99,
    'tau'                 : 1e-3,
    'max_train_per_client': 2000,
    'num_public'          : 250,
    'distill_epochs'      : 8,
    'temperature'         : 4.0,
}

LABEL_MAPS = {
    'PneumoniaMNIST' : {0: 0, 1: 1},
    'BreastMNIST'    : {0: 1, 1: 0},
    'DermaMNIST'     : {0: 1, 1: 1, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0},
    'PathMNIST'      : {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 1, 8: 1},
    'RetinaMNIST'    : {0: 0, 1: 1, 2: 1, 3: 1, 4: 1},
}
DATASET_LOADERS = {
    'PneumoniaMNIST' : PneumoniaMNIST,
    'BreastMNIST'    : BreastMNIST,
    'DermaMNIST'     : DermaMNIST,
    'PathMNIST'      : PathMNIST,
    'RetinaMNIST'    : RetinaMNIST,
}
CLASS_NAMES = ['Normal (0)', 'Abnormal (1)']

print('Configuration loaded')
for k, v in CONFIG.items(): print(f'   {k}: {v}')
print('\nAll 5 clients share the same 2 classes:', CLASS_NAMES)


Configuration loaded
   num_rounds: 8
   local_epochs: 4
   batch_size: 64
   learning_rate: 0.0005
   image_size: 96
   mu_prox: 0.01
   server_lr: 0.1
   beta1: 0.9
   beta2: 0.99
   tau: 0.001
   max_train_per_client: 2000
   num_public: 250
   distill_epochs: 8
   temperature: 4.0

All 5 clients share the same 2 classes: ['Normal (0)', 'Abnormal (1)']


---
## Section 3: Load and Relabel All 5 Datasets to the Shared Binary Task

In [ ]:
imagenet_mean, imagenet_std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
transform_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

class RelabeledMedMNIST(Dataset):
    def __init__(self, medmnist_dataset, label_map):
        self.ds = medmnist_dataset
        self.label_map = label_map
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        img, label = self.ds[i]
        native = int(label[0]) if hasattr(label, '__len__') else int(label)
        return img, self.label_map[native]

client_names, client_train_datasets, client_test_datasets = [], [], []

for name, LoaderClass in DATASET_LOADERS.items():
    train_raw = LoaderClass(split='train', transform=transform_train, download=True)
    test_raw  = LoaderClass(split='test',  transform=transform_test,  download=True)
    train_ds  = RelabeledMedMNIST(train_raw, LABEL_MAPS[name])
    test_ds   = RelabeledMedMNIST(test_raw,  LABEL_MAPS[name])

    if len(train_ds) > CONFIG['max_train_per_client']:
        idx = np.random.choice(len(train_ds), CONFIG['max_train_per_client'], replace=False)
        train_ds = Subset(train_ds, idx.tolist())

    client_names.append(name)
    client_train_datasets.append(train_ds)
    client_test_datasets.append(test_ds)

    labels_seen = [train_ds[i][1] for i in range(min(len(train_ds), 500))]
    n_abn = sum(labels_seen); n_norm = len(labels_seen) - n_abn
    print(f'   {name}: {len(train_ds)} train (capped) / {len(test_ds)} test '
          f'| sample balance in first 500: Normal={n_norm}, Abnormal={n_abn}')

client_sizes   = [len(ds) for ds in client_train_datasets]
client_loaders = [DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=True) for ds in client_train_datasets]
client_test_loaders = [DataLoader(ds, batch_size=64, shuffle=False) for ds in client_test_datasets]
test_loader_combined = DataLoader(ConcatDataset(client_test_datasets), batch_size=64, shuffle=False)

print(f'\nCombined test set across all 5 clients: {sum(len(d) for d in client_test_datasets)} images')

---
## Section 4: Shared Transfer Learning Model (frozen backbone + one shared binary head)

In [ ]:
class FederatedTransferModel(nn.Module):
    """
    FIXED VERSION: the previous version froze the ENTIRE ResNet-18 backbone and only trained a tiny
    2-layer head. This capped accuracy well below normal benchmark numbers (PneumoniaMNIST should reach
    roughly 85 to 90 percent, not the 40 to 57 percent seen before) because ImageNet features, learned from
    natural photographs, do not transfer well to grayscale medical scans without any adaptation.

    Fix: the backbone is now UNFROZEN and fine-tuned along with the head. Only the very first
    convolution and batch norm layer (the most generic, low level edge detectors) stay frozen; everything
    from layer1 onward is trainable and gets federated across clients, exactly like the head.
    """
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        for p in self.backbone.conv1.parameters():
            p.requires_grad = False
        for p in self.backbone.bn1.parameters():
            p.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)

_probe = FederatedTransferModel()
n_total     = sum(p.numel() for p in _probe.parameters())
n_trainable = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
print(f'Total parameters      : {n_total:,}')
print(f'Trainable (fine-tuned) : {n_trainable:,}  <- now includes most of the backbone, not just the head')
del _probe


---
## Section 5: Helper Functions

In [ ]:
def train_local(model, loader, optimizer, epochs, device):
    model.train()
    ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            ce(model(imgs), labels).backward()
            optimizer.step()

def train_local_fedprox(model, loader, optimizer, epochs, device, global_state, mu):
    model.train()
    ce = nn.CrossEntropyLoss()
    global_params = {k: v.clone().detach().to(device) for k, v in global_state.items()}
    for _ in range(epochs):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = ce(model(imgs), labels)
            prox = sum(((p - global_params[n]) ** 2).sum()
                       for n, p in model.named_parameters() if p.requires_grad and n in global_params)
            (loss + (mu / 2) * prox).backward()
            optimizer.step()

def evaluate_model(model, loader, device):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return correct / total

def get_soft_preds(model, public_X, device):
    model.eval()
    with torch.no_grad():
        return F.softmax(model(public_X.to(device)), dim=1).cpu()

def weighted_avg_trainable_state(models_list, sizes):
    """Averages every TRAINABLE parameter (backbone from layer1 onward, plus head) - not just the head
    anymore, since the backbone is now being fine-tuned and needs to be federated too."""
    total = sum(sizes)
    ref_model = models_list[0]
    trainable_keys = [n for n, p in ref_model.named_parameters() if p.requires_grad]
    state_dicts = [m.state_dict() for m in models_list]
    return {k: sum(state_dicts[i][k].float() * (sizes[i] / total) for i in range(len(models_list)))
            for k in trainable_keys}

def load_trainable_state(model, trainable_state):
    """Loads only the trainable parameters back into a model, leaving the frozen conv1/bn1 untouched."""
    full_state = model.state_dict()
    for k, v in trainable_state.items():
        full_state[k] = v
    model.load_state_dict(full_state)

print('Helper functions ready')


---
## Section 6: Run FedAvg, FedProx, FedAdam

In [ ]:
def run_weight_based_algo(algo_name, use_prox=False):
    label = 'FedProx' if use_prox else algo_name.upper()
    print('='*70); print(f'  {label}'); print('='*70)

    clients = [FederatedTransferModel().to(DEVICE) for _ in range(len(client_names))]
    opts    = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
               for m in clients]
    global_model = FederatedTransferModel().to(DEVICE)
    server_m, server_v = {}, {}
    history = {name: [] for name in client_names}
    history['round'] = []; history['combined'] = []

    for rnd in range(1, CONFIG['num_rounds'] + 1):
        g_trainable_state = {n: p.clone() for n, p in global_model.named_parameters() if p.requires_grad}
        for c in clients:
            load_trainable_state(c, g_trainable_state)

        for c, opt, loader in zip(clients, opts, client_loaders):
            if use_prox:
                train_local_fedprox(c, loader, opt, CONFIG['local_epochs'], DEVICE, g_trainable_state, CONFIG['mu_prox'])
            else:
                train_local(c, loader, opt, CONFIG['local_epochs'], DEVICE)

        avg_state = weighted_avg_trainable_state(clients, client_sizes)

        if algo_name == 'fedadam':
            beta1, beta2, tau, server_lr = CONFIG['beta1'], CONFIG['beta2'], CONFIG['tau'], CONFIG['server_lr']
            new_state = {}
            for key in avg_state:
                delta = avg_state[key].float() - g_trainable_state[key].float()
                if key not in server_m:
                    server_m[key] = torch.zeros_like(delta)
                    server_v[key] = torch.full_like(delta, tau ** 2)
                server_m[key] = beta1 * server_m[key] + (1 - beta1) * delta
                server_v[key] = beta2 * server_v[key] + (1 - beta2) * (delta ** 2)
                new_state[key] = g_trainable_state[key].float() + server_lr * server_m[key] / (torch.sqrt(server_v[key]) + tau)
            load_trainable_state(global_model, new_state)
        else:
            load_trainable_state(global_model, avg_state)

        combined_acc = evaluate_model(global_model, test_loader_combined, DEVICE)
        history['round'].append(rnd); history['combined'].append(combined_acc)
        round_accs = []
        for c, test_loader, name in zip(clients, client_test_loaders, client_names):
            acc = evaluate_model(global_model, test_loader, DEVICE)
            history[name].append(acc); round_accs.append(acc)
        acc_str = ' | '.join(f'{n}: {a:.4f}' for n, a in zip(client_names, round_accs))
        print(f'  Round {rnd:02d} | Combined: {combined_acc:.4f} | {acc_str}')

    print(f'\nFinal combined accuracy: {history["combined"][-1]:.4f}\n')
    return history

history_fedavg  = run_weight_based_algo('fedavg')
history_fedprox = run_weight_based_algo('fedavg', use_prox=True)
history_fedadam = run_weight_based_algo('fedadam')


---
## Section 7: Run FedDistillation (FedDF) - now usable again since labels are shared

In [ ]:
print('='*70); print('  FEDDISTILLATION (FedDF)'); print('='*70)

public_per_client = CONFIG['num_public'] // len(client_names)
public_pool = ConcatDataset([Subset(ds, range(min(public_per_client, len(ds)))) for ds in client_test_datasets])
public_loader = DataLoader(public_pool, batch_size=len(public_pool), shuffle=False)
public_X_fd, _ = next(iter(public_loader))

clients_fd = [FederatedTransferModel().to(DEVICE) for _ in range(len(client_names))]
opts_fd    = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['learning_rate'])
              for m in clients_fd]
global_fd  = FederatedTransferModel().to(DEVICE)
history_feddistill = {name: [] for name in client_names}
history_feddistill['round'] = []; history_feddistill['combined'] = []

for rnd in range(1, CONFIG['num_rounds'] + 1):
    for c, opt, loader in zip(clients_fd, opts_fd, client_loaders):
        train_local(c, loader, opt, CONFIG['local_epochs'], DEVICE)

    client_soft_preds = [get_soft_preds(c, public_X_fd, DEVICE) for c in clients_fd]
    weights = np.array(client_sizes) / sum(client_sizes)
    ensemble_soft = sum(w * p for w, p in zip(weights, client_soft_preds))

    global_opt = optim.Adam(filter(lambda p: p.requires_grad, global_fd.parameters()), lr=CONFIG['learning_rate'])
    T = CONFIG['temperature']
    for _ in range(CONFIG['distill_epochs']):
        global_opt.zero_grad()
        logits = global_fd(public_X_fd.to(DEVICE))
        s_log  = F.log_softmax(logits / T, dim=1)
        t_soft = F.softmax(ensemble_soft.to(DEVICE) / T, dim=1)
        (nn.KLDivLoss(reduction='batchmean')(s_log, t_soft) * (T ** 2)).backward()
        global_opt.step()

    for c in clients_fd:
        trainable_state = {n: p.clone() for n, p in global_fd.named_parameters() if p.requires_grad}
        load_trainable_state(c, trainable_state)

    combined_acc = evaluate_model(global_fd, test_loader_combined, DEVICE)
    history_feddistill['round'].append(rnd); history_feddistill['combined'].append(combined_acc)
    round_accs = []
    for test_loader, name in zip(client_test_loaders, client_names):
        acc = evaluate_model(global_fd, test_loader, DEVICE)
        history_feddistill[name].append(acc); round_accs.append(acc)
    acc_str = ' | '.join(f'{n}: {a:.4f}' for n, a in zip(client_names, round_accs))
    print(f'  Round {rnd:02d} | Combined: {combined_acc:.4f} | {acc_str}')

print(f'\nFinal combined accuracy: {history_feddistill["combined"][-1]:.4f}')


---
## Section 8: Compare All 4 Algorithms

In [ ]:
results = {
    'FedAvg'          : history_fedavg,
    'FedProx'         : history_fedprox,
    'FedAdam'         : history_fedadam,
    'FedDistillation' : history_feddistill,
}

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle('Federated Transfer Learning - 5 Medical Datasets, Shared Binary Normal/Abnormal Task',
             fontsize=13, fontweight='bold')
axes = axes.flatten()
palette = plt.cm.tab10(np.linspace(0, 1, 4))

for ax, name in zip(axes, client_names):
    for (algo, hist), color in zip(results.items(), palette):
        ax.plot(hist['round'], hist[name], '-o', lw=2, ms=4, color=color, label=algo)
    ax.set(xlabel='Round', ylabel='Test Accuracy', title=name)
    ax.legend(fontsize=7); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)

for (algo, hist), color in zip(results.items(), palette):
    axes[5].plot(hist['round'], hist['combined'], '-o', lw=2, ms=4, color=color, label=algo)
axes[5].set(xlabel='Round', ylabel='Combined Test Accuracy', title='Combined (All 5 Clients Pooled)')
axes[5].legend(fontsize=7); axes[5].grid(alpha=0.3); axes[5].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('binary_multi_medical_ftl_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved as binary_multi_medical_ftl_comparison.png')

print('\nFinal round accuracy - Algorithm x Client (and combined):')
header = f'{"Algorithm":16s} | ' + ' | '.join(f'{n[:13]:13s}' for n in client_names) + ' | Combined'
print(header); print('=' * len(header))
for algo, hist in results.items():
    row_vals = [hist[name][-1] for name in client_names]
    print(f'{algo:16s} | ' + ' | '.join(f'{v:13.4f}' for v in row_vals) + f' | {hist["combined"][-1]:.4f}')

---
## Section 9: Summary

**What changed from the previous version of this experiment:** every client's dataset is now relabeled into
the same two classes, Normal (0) and Abnormal (1), using clinically standard groupings for the multi-class
datasets (DermaMNIST, PathMNIST, RetinaMNIST). This removes the incompatible label space problem entirely,
brings back a single shared classification head across all 5 clients, and makes Federated Distillation
usable again, exactly as in the earlier 2 client version of this experiment.

**What to check in the results above:**
- Does any algorithm now achieve consistently strong accuracy across all 5 clients, rather than favoring
  one or two, the way FedAvg and FedProx tended to favor Pneumonia over Breast in the 2 client version?
- Does Federated Distillation still perform best here, as it did in the 2 client version, now that it has
  5 clients' worth of pooled proxy data instead of 2?
- Are some clients (for example RetinaMNIST or DermaMNIST, which involve harder or more subtle visual
  distinctions than Pneumonia's more visually obvious finding) simply harder for every algorithm, regardless
  of aggregation method?

**What to tell your professor:**
> *"Following your feedback, I simplified the 5 client medical experiment by relabeling every dataset into
> the same binary Normal versus Abnormal task, using standard clinical groupings for the datasets that were
> originally multi-class. This let me bring back a single shared classification head and compare all four
> algorithms, including Federated Distillation, directly and fairly across all 5 clients, the same way I did
> for the original 2 client Pneumonia and Breast experiment."*